# FDA Project — S&P 500 Bubble & Crisis Early-Warning System
## GSADF (Layer 1) → HMM (Layer 2) → Logit / Random Forest (Layer 3) → Bubble breadth of the S&P 500

| Proposal | Notebook นี้ |
|---|---|
| Layer 1: GSADF บน S&P 500 และ 11 GICS sectors | Part 2 — S&P 500 + 11 sector (รวม 48 อุตสาหกรรมของ Ken French ให้เป็น 11 กลุ่มแบบ GICS) + sector ไหนนำ |
| Layer 2: HMM Calm / Volatile / Crisis, transition, duration, persistence | Part 3 — BIC 2 vs 3 states, transition matrix, expected duration, persistence, เทียบ NBER และวิกฤตจริง, robustness ใส่ VIX |
| Layer 3: RF + Logistic baseline, หลาย horizon | Part 4–6 — ทาย P(bubble) ต่อ sector ที่ 6 / 12 / 24 เดือน; Logit (หลัก) vs Random Forest vs Logit+Macro |
| AUC + Precision-Recall เน้น Recall | Part 6 — AUC, PR-AUC, Recall / Precision, lead time, false alarm ต่อปี, paired bootstrap test |
| Test 2020 (COVID) และ 2022 (rates) | Part 7 — train ถึง 2008 แบบ proposal แล้ว test สองช่วงนี้ |
| SHAP แยกตามประเภทวิกฤต | Part 8 — SHAP (RF) + logit contribution: 2000 / 2008 / 2020 / 2022 |

**สิ่งที่เปลี่ยนจาก proposal และเหตุผล** (ผลจาก notebook S&P 500 รุ่นก่อน)
- Target ระดับ S&P 500 (ลง 10% ใน 1–6 เดือน) มีเหตุการณ์ไม่ถึง 10 ครั้ง → AUC คร่อม 0.5 ทุกช่อง
  จึงทายที่ระดับ **sector**: *P(bubble)* = โอกาสที่ sector ลง ≥ 40% ภายใน H เดือน (นิยามของ Greenwood, Shleifer & You, 2019)
  แล้วรวมกลับเป็นตัวชี้วัดของ S&P 500 (**Bubble breadth**, Part 9)
- Model หลักเป็น **Logit** (Random Forest ทายโอกาสต่ำเกินจริงช่วง run-up) — RF ยังอยู่เป็นตัวเปรียบเทียบตาม proposal
- HMM และ GSADF ใช้เป็น **regime / detection layer** ไม่ใช่ input ของ classifier (ใส่แล้วไม่ช่วยแยก sector และทำให้ข้อมูลช่วง 1930s หาย)
- Walk-forward ทุกปีตั้งแต่ 1960 เป็นผลหลัก (train 1997–2008 มีวิกฤตใหญ่แค่ 2 ครั้ง) และยังรายงานช่วง 2020 / 2022 แยก

## Part 0 — Set up

In [ ]:
%pip install -q hmmlearn shap yfinance xlrd

In [ ]:
import io, os, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from hmmlearn.hmm import GaussianHMM
import warnings
warnings.filterwarnings("ignore")
print("Libraries ready")

In [ ]:
# --- Global settings ---------------------------------------------------------
SEED         = 42
TRADING_DAYS = 252

# Layer 3 target: sector crash (Greenwood, Shleifer & You, 2019)
H_LIST    = [6, 12, 24]        # forecast horizons (months)
MAIN_H    = 24                 # horizon used for breadth, SHAP and the 2020/2022 charts
CRASH_DD  = 0.40               # crash = drawdown >= 40% within H months
RUNUP_2Y  = 1.00               # run-up: >= 100% in 24 months (raw and net of market)
RUNUP_5Y  = 0.50               #         and >= 50% in 60 months
MIN_FIRMS = 10                 # industry needs >= 10 firms
MASK_DD12 = -0.20              # month is "already falling" if >= 20% below its 12-month high

# Layer 1: GSADF / PSY
PSY_W0    = 36                 # minimum window (months)
PSY_TB    = 24                 # family-wise window for the critical value (months)
N_SIM_PSY = 99                 # wild-bootstrap replications

# Layer 2: HMM
HMM_START  = "1950-01-01"
VOL_WINDOW = 21                # days for realised volatility

# Walk-forward and evaluation
FIRST_TEST_YEAR    = 1960
MACRO_TEST_YEAR    = 1970      # first test year for the macro comparison (term spread starts 1953)
PROPOSAL_TRAIN_END = "2008-12-31"
TEST_WINDOWS = {"COVID-2020": ("2019-01-01", "2020-12-31"),
                "Rates-2022": ("2021-01-01", "2023-06-30")}
ALERT_MULT = 2.0               # alert when P(bubble) >= 2 x training base rate
N_BOOT     = 300               # block-bootstrap replications
BLOCK      = 24                # block length (months)

# Market level
MAIN_MODEL = "Logit"
HIGH_P     = 0.35              # "high bubble risk" sector
BEAR_DD    = 0.20              # S&P 500 bear market

REGIME_COLORS = {"Calm": "#2ecc71", "Volatile": "#f4a261", "Crisis": "#e74c3c"}
MODEL_COLORS  = {"Logit-Core": "#95a5a6", "Logit": "#1f4e79", "RandomForest": "#e74c3c", "Logit+Macro": "#f4a261"}
print("Settings ready")

## Part 1 — Data

| Data | Source | ใช้ใน |
|---|---|---|
| 48 industry portfolios: monthly / daily returns, number of firms, firm size, BE/ME | Ken French Data Library | Layer 1, 3 |
| Market return (Mkt-RF + RF) | Ken French | Net-of-market return |
| S&P 500 daily close, VIX | Yahoo Finance | Layer 1, 2, case study |
| BAA, AAA, GS10, TB3MS, USREC (NBER) | FRED | Macro features, validation |
| Shiller CAPE | Shiller data | Macro features |

In [ ]:
# --- Ken French data: download and parse ------------------------------------
FRENCH_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
FRENCH_FILES = {"ind":   "49_Industry_Portfolios_CSV.zip",
                "daily": "49_Industry_Portfolios_daily_CSV.zip",
                "ff":    "F-F_Research_Data_Factors_CSV.zip"}
LOCAL_DIR = "french_data"      # upload the three zip files here if the download fails

def load_french(name):
    fname = FRENCH_FILES[name]
    local = os.path.join(LOCAL_DIR, fname)
    if os.path.exists(local):
        raw = open(local, "rb").read()
    else:
        raw = requests.get(FRENCH_URL + fname, timeout=120, headers={"User-Agent": "Mozilla/5.0"}).content
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        return z.read(z.namelist()[0]).decode("latin-1")

def parse_french(text):
    # one CSV file holds several tables; each starts with a title line and a header line ",col1,col2,..."
    tables, title, last_text, cols, rows = {}, None, "", None, []

    def save_table():
        if cols is None or not rows:
            return
        idx = [r[0] for r in rows]
        df = pd.DataFrame([r[1:] for r in rows], columns=cols, index=idx).apply(pd.to_numeric, errors="coerce")
        df = df.mask(df <= -99.99)                                   # -99.99 = missing
        if len(idx[0]) == 6:
            df.index = pd.to_datetime(idx, format="%Y%m") + pd.offsets.MonthEnd(0)
        elif len(idx[0]) == 8:
            df.index = pd.to_datetime(idx, format="%Y%m%d")
        else:
            df.index = [int(i) for i in idx]
        tables[title if title not in tables else f"{title} ({len(tables)})"] = df

    for line in text.splitlines():
        s = line.strip()
        if s.startswith(","):
            save_table()
            title, cols, rows = last_text, [c.strip() for c in s.split(",")[1:]], []
        elif s[:1].isdigit() and cols is not None:
            parts = [p.strip() for p in s.split(",")]
            if parts[0].isdigit():
                rows.append(parts[:len(cols) + 1])
        else:
            if cols is not None and rows:
                save_table()
                cols, rows = None, []
            if s:
                last_text = s
    save_table()
    return tables

def get_table(tables, *words):
    for name, df in tables.items():
        if all(w.lower() in name.lower() for w in words):
            return df
    raise KeyError(f"Table with {words} not found")

ind_tables   = parse_french(load_french("ind"))
daily_tables = parse_french(load_french("daily"))
ff_tables    = parse_french(load_french("ff"))

drop_other = lambda df: df.drop(columns=["Other"], errors="ignore")    # 48 industries, as in the paper
ret    = drop_other(get_table(ind_tables, "value weighted", "monthly")) / 100
nfirms = drop_other(get_table(ind_tables, "number of firms")).reindex(ret.index)
fsize  = drop_other(get_table(ind_tables, "average firm size")).reindex(ret.index)
be_me  = drop_other(get_table(ind_tables, "sum of be", "sum of me"))
dret   = drop_other(get_table(daily_tables, "value weighted", "daily")) / 100
ff     = next(df for df in ff_tables.values() if isinstance(df.index, pd.DatetimeIndex))
mkt    = ((ff["Mkt-RF"] + ff["RF"]) / 100).reindex(ret.index)
mcap   = nfirms * fsize                                            # industry market cap ($ million)

INDS = list(ret.columns)
print(f"Industries   : {len(INDS)}")
print(f"Monthly data : {ret.index[0]:%Y-%m} -> {ret.index[-1]:%Y-%m}")
print(f"Daily data   : {dret.index[0]:%Y-%m-%d} -> {dret.index[-1]:%Y-%m-%d}")
ret.iloc[-3:, :8].round(4)

In [ ]:
# --- S&P 500 and VIX (Yahoo Finance) ----------------------------------------
import yfinance as yf

def yf_close(ticker, start):
    raw = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    return raw["Close"].dropna()

try:
    spx_daily = yf_close("^GSPC", "1927-01-01")
    vix_daily = yf_close("^VIX", "1990-01-01")
    SPX_NAME = "S&P 500"
except Exception as e:
    # fallback: cap-weighted market built from the industry data
    print(f"Yahoo Finance unavailable ({type(e).__name__}) -> market index built from industries")
    w = mcap.shift(1).div(mcap.shift(1).sum(axis=1), axis=0)
    w.index = w.index.to_period("M")
    w_d = w.reindex(dret.index.to_period("M")).set_axis(dret.index)
    spx_daily = (1 + (dret * w_d).sum(axis=1)).cumprod()
    vix_daily = None
    SPX_NAME = "US market"

spx_m = spx_daily.resample("ME").last()
print(f"{SPX_NAME} daily : {spx_daily.index[0].date()} -> {spx_daily.index[-1].date()} ({len(spx_daily):,} days)")
print(f"VIX daily        : {'not available' if vix_daily is None else f'{vix_daily.index[0].date()} -> {vix_daily.index[-1].date()}'}")

In [ ]:
# --- FRED and Shiller CAPE ----------------------------------------------------
def load_fred(series_id):
    try:
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
        s = pd.read_csv(url, index_col=0, parse_dates=True, na_values=".")
        s = s.iloc[:, 0].astype(float).dropna()
        s.index = s.index + pd.offsets.MonthEnd(0)                    # monthly value -> month end
        return s.rename(series_id)
    except Exception as e:
        print(f"FRED {series_id} unavailable ({type(e).__name__})")
        return None

SHILLER_SOURCES = ["ie_data.xls",
                   "https://img1.wsimg.com/blobby/go/e5e77e0b-59d1-44d9-ab25-4763ac982e53/downloads/ie_data.xls",
                   "http://www.econ.yale.edu/~shiller/data/ie_data.xls"]

def load_cape():
    for src in SHILLER_SOURCES:
        if not src.startswith("http") and not os.path.exists(src):
            continue
        try:
            x = pd.read_excel(src, sheet_name="Data", skiprows=7)
            x = x[["Date", "CAPE"]].apply(pd.to_numeric, errors="coerce").dropna()
            year = x["Date"].astype(int)
            month = ((x["Date"] - year) * 100).round().astype(int)
            idx = pd.to_datetime(dict(year=year, month=month, day=1)) + pd.offsets.MonthEnd(0)
            return pd.Series(x["CAPE"].values, index=idx, name="CAPE")
        except Exception as e:
            print(f"CAPE source failed ({type(e).__name__})")
    return None

fred = {sid: load_fred(sid) for sid in ["BAA", "AAA", "GS10", "TB3MS", "USREC"]}
cape = load_cape()
for sid, s in list(fred.items()) + [("CAPE", cape)]:
    print(f"{sid:6s}: {'not available' if s is None else f'{s.index[0]:%Y-%m} -> {s.index[-1]:%Y-%m}'}")

In [ ]:
# --- 11 GICS-like sectors from the 48 Fama-French industries ---------------
GICS_MAP = {
    "Energy":                 ["Coal", "Oil"],
    "Materials":              ["Chems", "Rubbr", "BldMt", "Steel", "Gold", "Mines", "Paper", "Boxes"],
    "Industrials":            ["Cnstr", "FabPr", "Mach", "ElcEq", "Aero", "Ships", "Guns", "BusSv", "Trans", "Whlsl"],
    "Consumer Discretionary": ["Toys", "Clths", "Txtls", "Autos", "PerSv", "Rtail", "Meals"],
    "Consumer Staples":       ["Agric", "Food", "Soda", "Beer", "Smoke", "Hshld"],
    "Health Care":            ["Hlth", "MedEq", "Drugs"],
    "Financials":             ["Banks", "Insur", "Fin"],
    "Information Technology": ["Hardw", "Softw", "Chips", "LabEq"],
    "Communication Services": ["Telcm", "Fun", "Books"],
    "Utilities":              ["Util"],
    "Real Estate":            ["RlEst"],
}
SECTOR_OF = {ind: sec for sec, inds in GICS_MAP.items() for ind in inds}
missing = [i for i in INDS if i not in SECTOR_OF]
print(f"Industries without a sector: {missing if missing else 'none'}")

# sector return = industry returns weighted by last month's market cap
w_prev = mcap.shift(1)
sector_ret = pd.DataFrame({
    sec: (ret[cols] * w_prev[cols]).sum(axis=1, min_count=1) / w_prev[cols].where(ret[cols].notna()).sum(axis=1)
    for sec, cols in ((s, [i for i in inds if i in INDS]) for s, inds in GICS_MAP.items())})
sector_ret = sector_ret.dropna(how="all")
print(f"Sector returns: {sector_ret.index[0]:%Y-%m} -> {sector_ret.index[-1]:%Y-%m}")
sector_ret.tail(3).round(4)

## Part 2 — Layer 1: Bubble detection (GSADF)
**PSY / GSADF** (Phillips, Shi & Yu, 2015):
- BSADF$_t$ = ค่าสูงสุดของ ADF t-stat จากทุก window ที่จบที่เดือน $t$ (ใช้ข้อมูลถึง $t$ เท่านั้น)
- GSADF = ค่าสูงสุดของ BSADF ทั้งช่วง → ทดสอบว่า "มี bubble อย่างน้อยหนึ่งช่วงหรือไม่"
- Critical value จาก **wild bootstrap** (Phillips & Shi, 2020) แบบ family-wise: คุมโอกาส flag ผิดใน 24 เดือนไว้ที่ 5%
- Bubble flag = BSADF > critical value **และ** ราคาขึ้นใน 12 เดือน (แยก bubble ออกจาก crash)

In [ ]:
# --- Helper functions: BSADF and wild-bootstrap critical values -------------
def bsadf_matrix(Y, w0=PSY_W0):
    # Y: (n_series, T) log prices without NaN. ADF regression dy_t = a + rho * y_{t-1} + e_t (lag 0)
    S, T = Y.shape
    x, z = Y[:, :-1], np.diff(Y, axis=1)
    csum = lambda a: np.concatenate([np.zeros((S, 1)), np.cumsum(a, axis=1)], axis=1)
    Sx, Sz, Sxx, Szz, Sxz = csum(x), csum(z), csum(x * x), csum(z * z), csum(x * z)
    out = np.full((S, T), np.nan)
    for j in range(w0 - 1, T - 1):                      # window end
        s = np.arange(0, j - w0 + 2)                    # every window start
        n = (j + 1 - s).astype(float)
        sx, sz = Sx[:, [j + 1]] - Sx[:, s], Sz[:, [j + 1]] - Sz[:, s]
        sxx = Sxx[:, [j + 1]] - Sxx[:, s]
        szz = Szz[:, [j + 1]] - Szz[:, s]
        sxz = Sxz[:, [j + 1]] - Sxz[:, s]
        vxx = np.maximum(sxx - sx * sx / n, 1e-12)
        cxz = sxz - sx * sz / n
        rho = cxz / vxx
        ssr = np.maximum((szz - sz * sz / n) - rho * cxz, 1e-12)
        out[:, j + 1] = np.max(rho / np.sqrt(ssr / (n - 2) / vxx), axis=1)
    return out

def run_psy(price, n_sim=N_SIM_PSY, tb=PSY_TB, seed=SEED):
    # returns a DataFrame (BSADF, CV, Bubble) and the GSADF test of the whole series
    logp = np.log(price.dropna())
    y = logp.values - logp.values[0]
    d = np.diff(y)
    e = d - np.cumsum(d) / np.arange(1, len(d) + 1)            # residuals under H0 (expanding mean)
    g = np.random.default_rng(seed).standard_normal((n_sim, len(d)))
    sims = np.concatenate([np.zeros((n_sim, 1)), np.cumsum(e * g, axis=1)], axis=1)
    b = bsadf_matrix(np.vstack([y, sims]))
    sim_max = pd.DataFrame(b[1:].T).rolling(tb, min_periods=1).max().values.T
    out = pd.DataFrame({"BSADF": b[0], "CV": np.nanquantile(sim_max, 0.95, axis=0)}, index=logp.index)
    rising = logp.diff(12) > 0
    out["Bubble"] = ((out["BSADF"] > out["CV"]) & rising).astype(float).where(out["CV"].notna())
    gsadf = {"GSADF": np.nanmax(b[0]), "CV 95%": np.nanquantile(np.nanmax(b[1:], axis=1), 0.95)}
    return out, gsadf

def episodes(flag, min_len=3):
    flag = flag.fillna(0).astype(bool)
    groups = (flag != flag.shift()).cumsum()
    rows = [(g.index[0], g.index[-1], len(g)) for _, g in flag.groupby(groups) if g.iloc[0] and len(g) >= min_len]
    return pd.DataFrame(rows, columns=["Start", "End", "Months"])

print("PSY helpers ready")

In [ ]:
# --- 2.1 GSADF on the S&P 500 (monthly) -------------------------------------
spx_psy, spx_gsadf = run_psy(spx_m)
spx_ep = episodes(spx_psy["Bubble"])

print(f"=== GSADF test — {SPX_NAME} ({spx_m.index[0]:%Y-%m} -> {spx_m.index[-1]:%Y-%m}) ===")
print(f"GSADF statistic : {spx_gsadf['GSADF']:.2f}")
print(f"Critical value  : {spx_gsadf['CV 95%']:.2f} (95%, wild bootstrap)")
print(f"Result          : {'REJECT H0 -> explosive episodes exist' if spx_gsadf['GSADF'] > spx_gsadf['CV 95%'] else 'no evidence of explosive behaviour'}")
print(f"\n=== Bubble episodes (>= 3 months) ===")
print(spx_ep.assign(Start=spx_ep["Start"].dt.strftime("%Y-%m"), End=spx_ep["End"].dt.strftime("%Y-%m")).to_string(index=False))

In [ ]:
# --- 2.2 GSADF on the 11 sectors --------------------------------------------
sector_px = (1 + sector_ret).cumprod()
sector_psy, rows = {}, []
for i, sec in enumerate(GICS_MAP):
    sector_psy[sec], g = run_psy(sector_px[sec], seed=SEED + i)
    rows.append({"Sector": sec, "GSADF": g["GSADF"], "CV 95%": g["CV 95%"],
                 "Reject H0": g["GSADF"] > g["CV 95%"],
                 "Bubble months": int(sector_psy[sec]["Bubble"].sum()),
                 "Episodes": len(episodes(sector_psy[sec]["Bubble"]))})
gsadf_table = pd.DataFrame(rows).set_index("Sector")
print("=== GSADF test by sector ===")
print(gsadf_table.round(2).to_string())

In [ ]:
# --- Visualise: GSADF bubble periods on the S&P 500 and the sectors ---------
fig, axes = plt.subplots(4, 3, figsize=(15, 13))
panels = [(SPX_NAME, spx_m, spx_psy)] + [(sec, sector_px[sec], sector_psy[sec]) for sec in GICS_MAP]
for ax, (name, px, sig) in zip(axes.ravel(), panels):
    px = px.loc["1950-01-01":]
    sig = sig.loc["1950-01-01":]
    ax.plot(px.index, px, color="#1f4e79", lw=0.9)
    ax.fill_between(sig.index, 0, 1, where=sig["Bubble"].eq(1), color="#e74c3c", alpha=0.35,
                    transform=ax.get_xaxis_transform())
    ax.set_yscale("log")
    ax.set_title(name, fontsize=10, fontweight="bold" if name == SPX_NAME else None)
    ax.grid(True, alpha=0.3)
fig.suptitle("Layer 1 — GSADF bubble periods (red) on the S&P 500 and the 11 sectors", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.3 Which sectors lead bubble formation? -------------------------------
LOOKBACK = 24   # months before the S&P 500 episode starts
rows = []
for _, ep in spx_ep.iterrows():
    row = {"S&P episode": f"{ep['Start']:%Y-%m} -> {ep['End']:%Y-%m}"}
    for sec in GICS_MAP:
        flag = sector_psy[sec]["Bubble"].loc[ep["Start"] - pd.DateOffset(months=LOOKBACK):ep["End"]]
        first = flag[flag == 1].index.min()
        row[sec] = (ep["Start"].to_period("M") - first.to_period("M")).n if pd.notna(first) else np.nan
    rows.append(row)
lead_table = pd.DataFrame(rows).set_index("S&P episode")

print(f"=== Months by which each sector's first bubble flag LEADS the S&P 500 episode (window: {LOOKBACK} months before) ===")
print("positive = sector flagged first | negative = after the S&P 500 | blank = no flag")
print(lead_table.to_string(na_rep=""))
leaders = (lead_table > 0).sum().reindex(list(GICS_MAP)).fillna(0).sort_values(ascending=False)
print(f"\nSectors that led most often: {', '.join(f'{s} ({int(n)})' for s, n in leaders.head(3).items())}")

In [ ]:
# --- 2.4 PSY flags for the 48 industries (used for the market-level breadth) -
ind_psy = pd.DataFrame(index=ret.index, columns=INDS, dtype=float)
for i, ind in enumerate(INDS):
    px = (1 + ret[ind].dropna()).cumprod()
    if len(px) > PSY_W0 + PSY_TB:
        ind_psy[ind] = run_psy(px, seed=SEED + 100 + i)[0]["Bubble"].reindex(ret.index)
print(f"Industry-months flagged as bubble: {ind_psy.stack().mean():.1%}")

## Part 3 — Layer 2: Market regimes (HMM)
Gaussian HMM บน **ผลตอบแทนรายวัน** และ **log realised volatility 21 วัน** ของ S&P 500 (ตั้งแต่ 1950)
- เลือก 2 หรือ 3 states ด้วย log-likelihood / BIC
- Transition matrix → expected duration = $1/(1-p_{ii})$ และ persistence $P(\text{ยังอยู่ใน regime เดิมอีก } k \text{ วัน})$
- Filtered probability $P(S_t \mid y_{1:t})$ (ใช้ข้อมูลถึงวันนั้นเท่านั้น) สำหรับ chart และ Part 6 / 9
- ประเมินเทียบ NBER recession และช่วงวิกฤตจริง (ตาม proposal)
- Robustness: ใส่ VIX (มีตั้งแต่ 1990)

In [ ]:
# --- HMM inputs and helper functions ----------------------------------------
hmm_df = pd.DataFrame({"Return": np.log(spx_daily).diff()})
hmm_df["Log_Vol"] = np.log(hmm_df["Return"].rolling(VOL_WINDOW).std() * np.sqrt(TRADING_DAYS))
hmm_df = hmm_df.loc[HMM_START:].dropna()
scaler_hmm = StandardScaler().fit(hmm_df)
X_hmm = scaler_hmm.transform(hmm_df)

def fit_hmm(X, n_states, seeds=range(3)):
    best, best_ll = None, -np.inf
    for s in seeds:
        m = GaussianHMM(n_components=n_states, covariance_type="full", n_iter=200, tol=1e-4, random_state=s)
        m.fit(X)
        ll = m.score(X)
        if ll > best_ll:
            best, best_ll = m, ll
    return best

def hmm_filter(model, X):
    # forward filter: P(state_t | data up to t)
    K = model.n_components
    log_b = np.column_stack([stats.multivariate_normal(model.means_[k], model.covars_[k],
                                                       allow_singular=True).logpdf(X) for k in range(K)])
    alpha = np.zeros((len(X), K))
    a = model.startprob_ * np.exp(log_b[0] - log_b[0].max())
    alpha[0] = a / a.sum()
    for t in range(1, len(X)):
        a = (alpha[t - 1] @ model.transmat_) * np.exp(log_b[t] - log_b[t].max())
        alpha[t] = a / a.sum()
    return alpha

def n_params(K, d):
    return (K - 1) + K * (K - 1) + K * d + K * d * (d + 1) / 2

print(f"HMM sample: {hmm_df.index[0].date()} -> {hmm_df.index[-1].date()} ({len(hmm_df):,} days)")

In [ ]:
# --- 3.1 How many regimes? 2 vs 3 states ------------------------------------
hmm_models, rows = {}, []
for K in [2, 3]:
    hmm_models[K] = fit_hmm(X_hmm, K)
    ll = hmm_models[K].score(X_hmm)
    p = n_params(K, X_hmm.shape[1])
    rows.append({"States": K, "Log-likelihood": ll, "Parameters": int(p),
                 "AIC": -2 * ll + 2 * p, "BIC": -2 * ll + p * np.log(len(X_hmm))})
sel = pd.DataFrame(rows).set_index("States")
print("=== HMM model selection (lower AIC / BIC = better) ===")
print(sel.round(1).to_string())
print(f"\nBIC prefers {sel['BIC'].idxmin()} states; the proposal uses 3 (Calm / Volatile / Crisis).")

In [ ]:
# --- 3.2 Transition matrix, expected duration and persistence (3 states) ----
hmm3 = hmm_models[3]
order = np.argsort(hmm3.means_[:, 1])                          # sort by volatility: Calm < Volatile < Crisis
REGIMES = ["Calm", "Volatile", "Crisis"]
A = pd.DataFrame(hmm3.transmat_[np.ix_(order, order)], index=REGIMES, columns=REGIMES)

state_means = pd.DataFrame(scaler_hmm.inverse_transform(hmm3.means_[order]), index=REGIMES, columns=hmm_df.columns)
duration = pd.DataFrame({
    "Ann. return (%)":      state_means["Return"] * TRADING_DAYS * 100,
    "Ann. volatility (%)":  np.exp(state_means["Log_Vol"]) * 100,
    "P(stay next day)":     np.diag(A),
    "Expected days":        1 / (1 - np.diag(A)),
    "P(same regime in 5d)":  [np.linalg.matrix_power(A.values, 5)[i, i] for i in range(3)],
    "P(same regime in 21d)": [np.linalg.matrix_power(A.values, 21)[i, i] for i in range(3)],
    "P(same regime in 63d)": [np.linalg.matrix_power(A.values, 63)[i, i] for i in range(3)],
}, index=REGIMES)

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(A, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Layer 2 — HMM transition matrix (daily)", fontsize=12, fontweight="bold")
ax.set_xlabel("To")
ax.set_ylabel("From")
plt.tight_layout()
plt.show()

print("=== Regime profile, expected duration and persistence ===")
print(duration.round(3).to_string())

In [ ]:
# --- 3.3 Filtered regimes over time -----------------------------------------
filt = pd.DataFrame(hmm_filter(hmm3, X_hmm)[:, order], index=hmm_df.index, columns=REGIMES)
regime = filt.idxmax(axis=1)
px = spx_daily.reindex(filt.index)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax = axes[0]
ax.plot(px.index, px, color="#1f4e79", lw=0.8)
for r, c in REGIME_COLORS.items():
    ax.fill_between(px.index, 0, 1, where=regime.eq(r), color=c, alpha=0.25, transform=ax.get_xaxis_transform(), label=r)
ax.set_yscale("log")
ax.set_ylabel(f"{SPX_NAME} (log)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
ax = axes[1]
ax.stackplot(filt.index, filt["Calm"], filt["Volatile"], filt["Crisis"], colors=list(REGIME_COLORS.values()), alpha=0.8,
             labels=["P(Calm)", "P(Volatile)", "P(Crisis)"])
ax.set_ylabel("Filtered probability")
ax.legend(loc="upper left", fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle("Layer 2 — HMM filtered regimes", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("Share of days in each regime:")
print(regime.value_counts(normalize=True).reindex(REGIMES).round(3).to_string())

In [ ]:
# --- 3.4 Validation: regimes vs NBER recessions and known crises ------------
KNOWN_CRISES = {"1973-74 oil shock": ("1973-01-11", "1974-10-03"),
                "1987 crash":        ("1987-08-25", "1987-12-04"),
                "2000-02 dot-com":   ("2000-03-24", "2002-10-09"),
                "2007-09 GFC":       ("2007-10-09", "2009-03-09"),
                "2020 COVID":        ("2020-02-19", "2020-03-23"),
                "2022 rates":        ("2022-01-03", "2022-10-12")}
rows = []
for name, (a, b) in KNOWN_CRISES.items():
    r = regime.loc[a:b]
    if len(r):
        rows.append({"Crisis (peak -> trough)": name, "Days": len(r),
                     "% Crisis regime": (r == "Crisis").mean() * 100,
                     "% Volatile or Crisis": r.isin(["Volatile", "Crisis"]).mean() * 100,
                     "First Crisis day": r[r == "Crisis"].index.min().date() if (r == "Crisis").any() else "none"})
crisis_check = pd.DataFrame(rows).set_index("Crisis (peak -> trough)")
print("=== HMM regimes inside known S&P 500 crises ===")
print(crisis_check.round(1).to_string())
print(f"\nUnconditional share of Crisis-regime days: {(regime == 'Crisis').mean():.1%}")

if fred["USREC"] is not None:
    rec = fred["USREC"].reindex(fred["USREC"].index.union(regime.index)).ffill().reindex(regime.index)
    tab = pd.crosstab(rec.map({1.0: "NBER recession", 0.0: "Expansion"}), regime, normalize="index").reindex(columns=REGIMES)
    print("\n=== Share of days in each regime: NBER recession vs expansion ===")
    print((tab * 100).round(1).to_string())
    is_crisis, in_rec = regime.eq("Crisis"), rec.eq(1)
    print(f"\nPrecision: P(recession | Crisis regime) = {in_rec[is_crisis].mean():.1%}")
    print(f"Recall   : P(Crisis regime | recession) = {is_crisis[in_rec].mean():.1%}")

In [ ]:
# --- 3.5 Robustness: add VIX as a third HMM input (1990 onwards) ------------
if vix_daily is not None:
    hv = hmm_df.join(np.log(vix_daily).rename("Log_VIX"), how="inner").dropna()
    sc_v = StandardScaler().fit(hv)
    hmm_vix = fit_hmm(sc_v.transform(hv), 3)
    o_v = np.argsort(hmm_vix.means_[:, 1])
    A_v = hmm_vix.transmat_[np.ix_(o_v, o_v)]
    reg_v = pd.Series(np.array(REGIMES)[hmm_filter(hmm_vix, sc_v.transform(hv))[:, o_v].argmax(axis=1)], index=hv.index)
    comp = pd.DataFrame({"Expected days (no VIX)": duration["Expected days"],
                         "Expected days (with VIX)": 1 / (1 - np.diag(A_v))}, index=REGIMES)
    print("=== Robustness: HMM with VIX (1990+) ===")
    print(comp.round(1).to_string())
    print(f"\nSame regime label on {(reg_v == regime.reindex(hv.index)).mean():.1%} of days since 1990")
else:
    print("VIX not available -> robustness check skipped")

In [ ]:
# --- 3.6 Monthly regime series (used in Part 6 and Part 9) ------------------
regime_m = filt.resample("ME").mean()
regime_m["Regime"] = regime_m[REGIMES].idxmax(axis=1)
print(regime_m["Regime"].value_counts().reindex(REGIMES).to_string())

## Part 4 — Layer 3 data: features and labels

| Feature | นิยาม | กลุ่ม |
|---|---|---|
| `RunUp` | ขึ้น ≥100% ใน 24 เดือน (raw และ net-of-market) และ ≥50% ใน 60 เดือน | Core (paper) |
| `Net24` | ผลตอบแทน 24 เดือน − ตลาด | Core |
| `Vol_pre` | ความผันผวนรายเดือน 12 เดือนก่อนช่วง run-up | Core |
| `Accel` | ผลตอบแทน 24 เดือน − ผลตอบแทนปีแรกของช่วงนั้น (ราคาเร่งขึ้นช่วงหลัง) | Characteristic |
| `Vol_Rank`, `Vol_Chg` | rank ของ volatility 6 เดือนเทียบอุตสาหกรรมอื่น, การเปลี่ยนแปลงจากปีก่อน | Characteristic |
| `Firms_Chg24` | การเปลี่ยนแปลงจำนวนบริษัท (proxy ของ IPO / issuance) | Characteristic |
| `BM_Rank` | rank ของ book-to-market | Characteristic |
| `Credit_Spread`, `Credit_Chg12`, `Term_Spread`, `CAPE_Pctl`, `Sector_Disp` | macro-financial indicators ใน proposal | Macro (ablation) |

**Label** `Crash_H` = 1 ถ้า sector ลง ≥40% จากจุดสูงสุดใดๆ ภายใน H เดือน (H = 6, 12, 24)
**Mask** = เดือนที่ sector ต่ำกว่า peak 12 เดือน ≥20% ("แตกไปแล้ว") — ไม่ใช้ทั้ง train และ test ของ early warning

In [ ]:
# --- 4.1 Industry features ---------------------------------------------------
log_p   = np.log1p(ret).cumsum()
log_mkt = np.log1p(mkt).cumsum()
past    = lambda lp, k: np.expm1(lp - lp.shift(k))

R24   = past(log_p, 24)
R60   = past(log_p, 60)
Net24 = R24.sub(past(log_mkt, 24), axis=0)
Accel = R24 - np.expm1(log_p.shift(12) - log_p.shift(24))
RunUp = ((R24 >= RUNUP_2Y) & (Net24 >= RUNUP_2Y) & (R60 >= RUNUP_5Y)).astype(float).where(R60.notna())
Vol_pre = ret.shift(24).rolling(12, min_periods=10).std() * np.sqrt(12)

price = np.exp(log_p)
DD12  = price / price.rolling(12, min_periods=12).max() - 1

vol_d = dret.groupby(dret.index.to_period("M")).std() * np.sqrt(TRADING_DAYS)
vol_d.index = vol_d.index.to_timestamp("M")
vol_d = vol_d.reindex(ret.index)
Vol6     = vol_d.rolling(6, min_periods=4).mean()
Vol_Rank = Vol6.rank(axis=1, pct=True)
Vol_Chg  = np.log(Vol6 / vol_d.shift(12).rolling(12, min_periods=8).mean())

Firms_Chg24 = np.log(nfirms / nfirms.shift(24))

bm = be_me.copy()
bm.index = pd.to_datetime([f"{y}-12-31" for y in bm.index])          # BE/ME of year Y usable from December Y
BM_Rank = bm.reindex(bm.index.union(ret.index)).ffill().reindex(ret.index).rank(axis=1, pct=True)
print("Industry features ready")

In [ ]:
# --- 4.2 Labels: crash within H months ---------------------------------------
def crash_label(price, H, dd=CRASH_DD):
    run_max = price.copy()
    max_dd = pd.DataFrame(0.0, index=price.index, columns=price.columns)
    for k in range(1, H + 1):
        if k > 1:
            run_max = np.fmax(run_max, price.shift(-(k - 1)))
        max_dd = np.fmax(max_dd, 1 - price.shift(-k) / run_max)
    complete = price.shift(-H).notna() & price.notna()
    return (max_dd >= dd).astype(float).where(complete)

CRASH = {H: crash_label(price, H) for H in H_LIST}

# crash events: first month a sector is >= 40% below its 24-month high (new event needs 24 months gap)
dd24 = price / price.rolling(24, min_periods=1).max() - 1
event_rows = []
for ind in INDS:
    hit = dd24[ind] <= -CRASH_DD
    last = None
    for d in hit[hit].index:
        if last is None or (d.to_period("M") - last.to_period("M")).n > 24:
            event_rows.append({"Industry": ind, "Onset": d})
            last = d
EVENTS = pd.DataFrame(event_rows)

rows = [{"Horizon": f"{H}M", "Base rate (all months)": CRASH[H].stack().mean()} for H in H_LIST]
print("=== Label base rates ===")
print(pd.DataFrame(rows).set_index("Horizon").round(3).to_string())
print(f"\nSector crash events (>= {CRASH_DD:.0%} below 24-month high): {len(EVENTS)} "
      f"({EVENTS['Onset'].min():%Y} -> {EVENTS['Onset'].max():%Y})")

In [ ]:
# --- 4.3 Label cross-check with NBER recessions (proposal) ------------------
if fred["USREC"] is not None:
    rec_m = fred["USREC"].reindex(ret.index).ffill()
    near = rec_m.rolling(25, center=True, min_periods=1).max()            # recession within +/- 12 months
    ev = EVENTS[EVENTS["Onset"] >= rec_m.first_valid_index()].copy()
    ev["In recession"] = ev["Onset"].map(rec_m).eq(1)
    ev["Within 12M of recession"] = ev["Onset"].map(near).eq(1)
    by_year = ev.groupby(ev["Onset"].dt.year).size()
    print("=== Sector crash events vs NBER recessions ===")
    print(f"Events                               : {len(ev)}")
    print(f"Onset during a recession             : {ev['In recession'].mean():.1%} (share of months in recession: {rec_m.mean():.1%})")
    print(f"Onset within 12 months of a recession: {ev['Within 12M of recession'].mean():.1%} (share of months: {near.mean():.1%})")
    print(f"Years with the most crash onsets     : {', '.join(f'{y} ({n})' for y, n in by_year.sort_values(ascending=False).head(5).items())}")
else:
    print("USREC not available -> NBER cross-check skipped")

In [ ]:
# --- 4.4 Macro-financial features (proposal Layer 3 inputs) -----------------
def monthly_lagged(s, lag=1):
    # value of month M is usable at the end of month M + lag
    if s is None:
        return pd.Series(np.nan, index=ret.index)
    s = s.copy()
    s.index = s.index + pd.offsets.MonthEnd(lag)
    return s.reindex(s.index.union(ret.index)).ffill().reindex(ret.index)

macro = pd.DataFrame(index=ret.index)
if fred["BAA"] is not None and fred["AAA"] is not None:
    macro["Credit_Spread"] = monthly_lagged(fred["BAA"] - fred["AAA"])
    macro["Credit_Chg12"]  = macro["Credit_Spread"].diff(12)
if fred["GS10"] is not None and fred["TB3MS"] is not None:
    macro["Term_Spread"] = monthly_lagged((fred["GS10"] - fred["TB3MS"]).dropna())
if cape is not None:
    cape_pctl = cape.expanding(min_periods=120).apply(lambda s: (s <= s[-1]).mean(), raw=True)
    macro["CAPE_Pctl"] = monthly_lagged(cape_pctl, lag=3)                 # Shiller earnings arrive with a lag
macro["Sector_Disp"] = past(log_p, 12).std(axis=1)                        # dispersion of 12M industry returns
MACRO_FEATS = [c for c in macro.columns if macro[c].notna().any()]
print(f"Macro features available: {MACRO_FEATS}")
print(macro.dropna().tail(3).round(3).to_string())

In [ ]:
# --- 4.5 Panel: one row per industry-month ----------------------------------
FEATS_CORE = ["RunUp", "Net24", "Vol_pre"]
FEATS_MAIN = FEATS_CORE + ["Accel", "Vol_Rank", "Vol_Chg", "Firms_Chg24", "BM_Rank"]
FEATS_MACRO = FEATS_MAIN + MACRO_FEATS

def to_long(wide):
    wide = wide.reindex(index=ret.index, columns=INDS)
    idx = pd.MultiIndex.from_product([wide.index, wide.columns], names=["Date", "Industry"])
    return pd.Series(wide.values.ravel(), index=idx)

panel = pd.concat({"RunUp": to_long(RunUp), "Net24": to_long(Net24), "Vol_pre": to_long(Vol_pre),
                   "Accel": to_long(Accel), "Vol_Rank": to_long(Vol_Rank), "Vol_Chg": to_long(Vol_Chg),
                   "Firms_Chg24": to_long(Firms_Chg24), "BM_Rank": to_long(BM_Rank), "R24": to_long(R24),
                   "DD12": to_long(DD12), "NFirms": to_long(nfirms), "MCap": to_long(mcap),
                   "PSY_Bubble": to_long(ind_psy),
                   **{f"Crash_{H}": to_long(CRASH[H]) for H in H_LIST}}, axis=1)
dates = panel.index.get_level_values("Date")
for c in MACRO_FEATS:
    panel[c] = macro[c].reindex(dates).values
panel = panel[panel["NFirms"] >= MIN_FIRMS]
panel["Masked"] = panel["DD12"] <= MASK_DD12
panel = panel.dropna(subset=FEATS_MAIN)
panel["Year"] = panel.index.get_level_values("Date").year

print(f"Panel rows  : {len(panel):,} industry-months ({panel['Year'].min()} -> {panel['Year'].max()})")
print(f"Masked rows : {panel['Masked'].mean():.1%} (already >= 20% below the 12-month high)")
panel[FEATS_MAIN].describe().T.round(3)

## Part 5 — Layer 3: Walk-forward models
- Refit ทุกปีตั้งแต่ 1960 (expanding window, train เริ่ม 1931)
- **Purge**: train ใช้เฉพาะเดือนที่ผลของ label รู้แล้ว ณ สิ้นปีก่อนหน้า ($t + H \le$ train end)
- Train และประเมินผลหลักบนเดือน early-warning (ไม่ masked) · ทายถึงเดือนล่าสุดสำหรับ Part 9

| Model | Inputs | บทบาท |
|---|---|---|
| `Base rate` | ค่าเฉลี่ยใน training | benchmark |
| `Logit-Core` | RunUp, Net24, Vol_pre | สมการ (1) ของ Greenwood et al. |
| **`Logit`** | Core + characteristics | **model หลัก** |
| `RandomForest` | เหมือน `Logit` | non-linear (proposal) |
| `Logit+Macro` | `Logit` + macro-financial | proposal features (ablation, test 1970+) |

In [ ]:
# --- Model definitions -------------------------------------------------------
def make_model(kind):
    if kind == "rf":
        return RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=300, max_samples=0.5,
                                      n_jobs=-1, random_state=SEED)
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))

MODEL_SPECS = {"Logit-Core":   ("logit", FEATS_CORE),
               "Logit":        ("logit", FEATS_MAIN),
               "RandomForest": ("rf",    FEATS_MAIN),
               "Logit+Macro":  ("logit", FEATS_MACRO)}
MODELS = list(MODEL_SPECS)
EXPLAIN_YEARS = {"Dot-com 2000": 1999, "GFC 2008": 2007, "COVID 2020": 2019, "Rates 2022": 2021}
print(f"Models: {MODELS}")

In [ ]:
# --- Walk-forward function ---------------------------------------------------
def run_walk_forward(H, first_year=FIRST_TEST_YEAR, keep_years=()):
    label = f"Crash_{H}"
    labelled = panel.dropna(subset=[label])
    preds, kept = [], {}
    for year in range(first_year, panel["Year"].max() + 1):
        train_end = pd.Timestamp(f"{year - 1}-12-31")
        tr_dates = labelled.index.get_level_values("Date")
        train = labelled[(tr_dates <= train_end - pd.DateOffset(months=H)) & ~labelled["Masked"]]
        test = panel[panel["Year"] == year]
        if test.empty or train[label].nunique() < 2:
            continue
        out = test[[label, "Masked", "MCap", "RunUp", "PSY_Bubble"]].rename(columns={label: "y"})
        out["Base rate"] = train[label].mean()
        for name, (kind, feats) in MODEL_SPECS.items():
            tr = train.dropna(subset=feats)
            ok = test[feats].notna().all(axis=1)
            out[name] = np.nan
            if len(tr) < 1000 or tr[label].nunique() < 2 or not ok.any():
                continue
            model = make_model(kind).fit(tr[feats], tr[label].astype(int))
            out.loc[ok, name] = model.predict_proba(test.loc[ok, feats])[:, 1]
            if year in keep_years and name in ("Logit", "RandomForest"):
                kept[(year, name)] = (model, tr[feats])
        preds.append(out)
    return pd.concat(preds).sort_index(), kept

PRED, FITTED = {}, {}
for H in H_LIST:
    PRED[H], kept = run_walk_forward(H, keep_years=set(EXPLAIN_YEARS.values()) if H == MAIN_H else ())
    FITTED.update(kept)
    p = PRED[H]
    print(f"H = {H:2d}M: {p.index.get_level_values('Date').min():%Y-%m} -> {p.index.get_level_values('Date').max():%Y-%m}, "
          f"{p['y'].notna().sum():,} labelled rows")

## Part 6 — Evaluation (out-of-sample, early-warning months)
- **AUC-ROC** และ **PR-AUC** (proposal) · PR lift = PR-AUC ÷ base rate (1 = สุ่ม)
- **XS-AUC** = AUC ภายในเดือนเดียวกัน → แยก sector ที่เสี่ยงได้ไหม (ตัดผลของ "ช่วงเวลา" ออก)
- **BSS** = 1 − Brier / Brier(base rate): > 0 แปลว่า probability ใกล้ความจริงกว่าการตอบ base rate
- **Recall / Precision** ที่ threshold "P ≥ 2 × base rate" (proposal เน้น Recall)
- CI 95% = block bootstrap ตามเวลา (บล็อก 24 เดือน, ทุก sector ในเดือนเดียวกันติดไปด้วย)

In [ ]:
# --- Helper functions: metrics and bootstrap --------------------------------
def month_blocks(df, n_boot=N_BOOT, block=BLOCK, seed=SEED):
    rng = np.random.default_rng(seed)
    months = df.index.get_level_values("Date").to_period("M")
    uniq = months.unique().sort_values()
    rows_of = pd.Series(np.arange(len(df))).groupby(months.values).apply(lambda s: s.values).reindex(uniq)
    for _ in range(n_boot):
        starts = rng.integers(0, len(uniq) - block + 1, size=int(np.ceil(len(uniq) / block)))
        picked = np.concatenate([np.arange(s, s + block) for s in starts])[:len(uniq)]
        yield np.concatenate([rows_of.iloc[i] for i in picked])

def xs_auc(df, col):
    aucs = [roc_auc_score(g["y"], g[col]) for _, g in df.groupby(level="Date")
            if g["y"].nunique() == 2 and g[col].nunique() > 1]
    return np.mean(aucs) if aucs else np.nan

def bss(y, p, base):
    return 1 - np.mean((p - y) ** 2) / np.mean((base - y) ** 2)

def evaluate(df, models):
    df = df.dropna(subset=["y"] + models)
    y, base = df["y"].values.astype(int), df["Base rate"].values
    rows = {}
    for m in models:
        p = df[m].values
        alert = p >= ALERT_MULT * base
        rows[m] = {"N": len(y), "Crash rate": y.mean(),
                   "AUC": roc_auc_score(y, p), "XS-AUC": xs_auc(df, m),
                   "PR-AUC": average_precision_score(y, p), "PR lift": average_precision_score(y, p) / y.mean(),
                   "BSS": bss(y, p, base), "Mean p": p.mean(),
                   "Recall": alert[y == 1].mean(), "Precision": y[alert].mean() if alert.any() else np.nan,
                   "Alert rate": alert.mean()}
    draws = {m: [] for m in models}
    for ix in month_blocks(df):
        yb = y[ix]
        if yb.min() == yb.max():
            continue
        for m in models:
            draws[m].append((roc_auc_score(yb, df[m].values[ix]), bss(yb, df[m].values[ix], base[ix])))
    for m in models:
        d = np.array(draws[m])
        rows[m].update({"AUC lo": np.percentile(d[:, 0], 2.5), "AUC hi": np.percentile(d[:, 0], 97.5),
                        "BSS lo": np.percentile(d[:, 1], 2.5), "BSS hi": np.percentile(d[:, 1], 97.5)})
    return pd.DataFrame(rows).T

def paired_test(df, a, b):
    df = df.dropna(subset=["y", a, b])
    y, base = df["y"].values.astype(int), df["Base rate"].values
    diff = lambda ix: (roc_auc_score(y[ix], df[a].values[ix]) - roc_auc_score(y[ix], df[b].values[ix]),
                       bss(y[ix], df[a].values[ix], base[ix]) - bss(y[ix], df[b].values[ix], base[ix]))
    point = diff(np.arange(len(y)))
    d = np.array([diff(ix) for ix in month_blocks(df) if y[ix].min() != y[ix].max()])
    return {"AUC diff": point[0], "AUC lo": np.percentile(d[:, 0], 2.5), "AUC hi": np.percentile(d[:, 0], 97.5),
            "p (AUC: a <= b)": (d[:, 0] <= 0).mean(),
            "BSS diff": point[1], "BSS lo": np.percentile(d[:, 1], 2.5), "BSS hi": np.percentile(d[:, 1], 97.5),
            "p (BSS: a <= b)": (d[:, 1] <= 0).mean()}

def early_warning(H):
    p = PRED[H]
    return p[~p["Masked"].astype(bool) & p["y"].notna()]

print("Evaluation helpers ready")

In [ ]:
# --- 6.1 Main results: every horizon ----------------------------------------
BASE_MODELS = ["Logit-Core", "Logit", "RandomForest"]
results = {H: evaluate(early_warning(H), ["Base rate"] + BASE_MODELS) for H in H_LIST}

for H in H_LIST:
    r = results[H]
    print(f"=== H = {H}M | {int(r['N'].iloc[0]):,} early-warning sector-months, crash rate {r['Crash rate'].iloc[0]:.3f} ===")
    print("-- Discrimination --")
    print(r[["AUC", "AUC lo", "AUC hi", "XS-AUC", "PR-AUC", "PR lift"]].round(3).to_string())
    print("-- Probability and alerts (alert = P >= 2 x base rate) --")
    print(r[["BSS", "BSS lo", "BSS hi", "Mean p", "Recall", "Precision", "Alert rate"]].round(3).to_string())
    print()

In [ ]:
# --- Visualise: accuracy / lead-time trade-off across horizons --------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["XS-AUC", "PR lift", "BSS"]):
    for m in BASE_MODELS:
        ax.plot([f"{H}M" for H in H_LIST], [results[H].loc[m, metric] for H in H_LIST], "o-",
                color=MODEL_COLORS[m], label=m)
    ax.axhline(0.5 if metric == "XS-AUC" else (1 if metric == "PR lift" else 0), color="black", ls="--", lw=0.8)
    ax.set_title(metric)
    ax.set_xlabel("Forecast horizon")
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=8)
fig.suptitle("Layer 3 — skill by forecast horizon (dashed = no skill)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.2 Paired tests: does non-linearity or macro data help? ---------------
rows = []
for H in H_LIST:
    ew = early_warning(H)
    rows.append({"Horizon": f"{H}M", "Comparison": "RandomForest - Logit", **paired_test(ew, "RandomForest", "Logit")})
    rows.append({"Horizon": f"{H}M", "Comparison": "Logit - Logit-Core", **paired_test(ew, "Logit", "Logit-Core")})
    if MACRO_FEATS:
        ew70 = ew[ew.index.get_level_values("Date").year >= MACRO_TEST_YEAR]
        rows.append({"Horizon": f"{H}M", "Comparison": f"Logit+Macro - Logit ({MACRO_TEST_YEAR}+)",
                     **paired_test(ew70, "Logit+Macro", "Logit")})
tests = pd.DataFrame(rows).set_index(["Horizon", "Comparison"])
print("=== Paired block-bootstrap tests (positive diff = first model better) ===")
print(tests.round(3).to_string())
print("\nRead: CI entirely above 0 -> first model significantly better; CI crossing 0 -> no evidence of a difference")

In [ ]:
# --- 6.3 Reliability: is a predicted 30% really about 30%? (main horizon) ---
ew = early_warning(MAIN_H)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, m in zip(axes, BASE_MODELS):
    q = pd.qcut(ew[m].rank(method="first"), 10, labels=False)
    rel = ew.groupby(q).agg(p=(m, "mean"), obs=("y", "mean"))
    lim = max(rel["p"].max(), rel["obs"].max()) * 1.1
    ax.plot([0, lim], [0, lim], "k--", lw=1)
    ax.plot(rel["p"], rel["obs"], "o-", color=MODEL_COLORS[m])
    ax.set_title(m)
    ax.set_xlabel("Mean predicted probability (decile)")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("Observed crash frequency")
fig.suptitle(f"Reliability — {MAIN_H}M horizon, out-of-sample", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

ru = ew[ew["RunUp"] == 1]
print(f"After a run-up ({len(ru)} sector-months): realised crash rate {ru['y'].mean():.1%} | "
      + " | ".join(f"{m} {ru[m].mean():.1%}" for m in BASE_MODELS))

In [ ]:
# --- 6.4 Event level: detection, lead time and false alarms -----------------
rows = []
for H in H_LIST:
    p = PRED[H]
    p_dates = p.index.get_level_values("Date")
    first_test = pd.Timestamp(f"{FIRST_TEST_YEAR}-01-01") + pd.DateOffset(months=H)
    last_lab = p.loc[p["y"].notna()].index.get_level_values("Date").max()
    ev = EVENTS[(EVENTS["Onset"] >= first_test) & (EVENTS["Onset"] <= last_lab)]
    n_years = (last_lab - pd.Timestamp(f"{FIRST_TEST_YEAR}-01-01")).days / 365.25
    for m in BASE_MODELS:
        alert = (p[m] >= ALERT_MULT * p["Base rate"]) & ~p["Masked"].astype(bool)
        alert_of = {ind: a.droplevel("Industry") for ind, a in alert.groupby(level="Industry")}
        leads = []
        for _, e in ev.iterrows():
            if e["Industry"] not in alert_of:
                continue
            a = alert_of[e["Industry"]].loc[e["Onset"] - pd.DateOffset(months=H):e["Onset"] - pd.DateOffset(months=1)]
            if a.any():
                leads.append((e["Onset"].to_period("M") - a[a].index.min().to_period("M")).n)
        # false alarm = start of an alert run with no crash onset in that industry within the next H months
        false_runs = 0
        for ind, a in alert_of.items():
            a = a.loc[:last_lab]
            starts = a.index[a & ~a.shift(fill_value=False)]
            onsets = ev.loc[ev["Industry"] == ind, "Onset"]
            false_runs += sum(not ((onsets > s) & (onsets <= s + pd.DateOffset(months=H))).any() for s in starts)
        rows.append({"Horizon": f"{H}M", "Model": m, "Crash events": len(ev), "Detected": len(leads),
                     "Detection rate": len(leads) / max(len(ev), 1),
                     "Median lead (months)": np.median(leads) if leads else np.nan,
                     "False alarms / year": false_runs / n_years,
                     "Share of months on alert": alert.loc[p_dates <= last_lab].mean()})
event_table = pd.DataFrame(rows).set_index(["Horizon", "Model"])
print(f"=== Event-level early warning (alert = P >= {ALERT_MULT:.0f} x base rate, sector not already falling) ===")
print(event_table.round(2).to_string())

In [ ]:
# --- 6.5 Results by HMM market regime (Layer 2 x Layer 3) -------------------
ew = early_warning(MAIN_H).copy()
ew["Regime"] = regime_m["Regime"].reindex(ew.index.get_level_values("Date")).values
rows = []
for r in REGIMES:
    g = ew[ew["Regime"] == r]
    if len(g) and g["y"].nunique() == 2:
        rows.append({"Regime": r, "Sector-months": len(g), "Crash rate": g["y"].mean(),
                     f"Mean P ({MAIN_MODEL})": g[MAIN_MODEL].mean(), "AUC": roc_auc_score(g["y"], g[MAIN_MODEL]),
                     "XS-AUC": xs_auc(g, MAIN_MODEL)})
print(f"=== {MAIN_MODEL}, {MAIN_H}M horizon, by HMM market regime ===")
print(pd.DataFrame(rows).set_index("Regime").round(3).to_string())

## Part 7 — Proposal test windows: COVID-2020 and Rates-2022
Train ครั้งเดียวด้วยข้อมูลถึง **2008** (ตาม proposal) แล้ว test สองช่วง · เทียบกับ model ที่ refit ทุกปี (Part 5)

In [ ]:
# --- 7.1 Fit once on data up to 2008, test 2019-2023 ------------------------
rows, PRED_2008 = [], {}
for H in H_LIST:
    label = f"Crash_{H}"
    lab = panel.dropna(subset=[label])
    train = lab[(lab.index.get_level_values("Date") <= pd.Timestamp(PROPOSAL_TRAIN_END) - pd.DateOffset(months=H))
                & ~lab["Masked"]]
    test = panel.loc[panel["Year"] >= 2019]
    out = test[[label, "Masked", "MCap", "RunUp"]].rename(columns={label: "y"})
    out["Base rate"] = train[label].mean()
    for name in BASE_MODELS:
        kind, feats = MODEL_SPECS[name]
        out[name] = make_model(kind).fit(train[feats], train[label].astype(int)).predict_proba(test[feats])[:, 1]
    PRED_2008[H] = out
    for win, (a, b) in TEST_WINDOWS.items():
        w = out.loc[pd.IndexSlice[a:b, :], :]
        w = w[~w["Masked"].astype(bool) & w["y"].notna()]
        if w["y"].nunique() < 2:
            continue
        for m in BASE_MODELS:
            alert = w[m] >= ALERT_MULT * w["Base rate"]
            rows.append({"Window": win, "Horizon": f"{H}M", "Model": m, "N": len(w), "Crash rate": w["y"].mean(),
                         "AUC": roc_auc_score(w["y"], w[m]), "PR-AUC": average_precision_score(w["y"], w[m]),
                         "Recall": alert[w["y"] == 1].mean(), "Precision": w["y"][alert].mean() if alert.any() else np.nan,
                         "BSS": bss(w["y"].values, w[m].values, w["Base rate"].values)})
window_table = pd.DataFrame(rows).set_index(["Window", "Horizon", "Model"])
print(f"=== Proposal test windows (trained on data up to {PROPOSAL_TRAIN_END[:4]}) ===")
print(window_table.round(3).to_string())

In [ ]:
# --- 7.2 Signal timing vs drawdown onset (2020 and 2022) --------------------
def breadth_series(pred, model):
    w = pred["MCap"] / pred.groupby(level="Date")["MCap"].transform("sum")
    keep = ~pred["Masked"].astype(bool)
    return (w * pred[model] * keep).groupby(level="Date").sum()

def drawdown_onset(start, end, level=-0.10):
    # first day in the window when the S&P 500 falls through 10% below its previous high
    dd = (spx_daily / spx_daily.cummax() - 1).loc[start:end]
    cross = (dd <= level) & (dd.shift(1) > level)
    return cross[cross].index.min()

fig, axes = plt.subplots(2, 2, figsize=(15, 7), sharex="col")
for col, (win, (a, b)) in enumerate(TEST_WINDOWS.items()):
    start = pd.Timestamp(a) - pd.DateOffset(months=6)
    px = spx_m.loc[start:b]
    onset = drawdown_onset(a, b)
    ax = axes[0, col]
    ax.plot(px.index, px, color="#1f4e79")
    if pd.notna(onset):
        ax.axvline(onset, color="#e74c3c", ls="--", label=f"Drawdown >= 10%: {onset:%Y-%m-%d}")
    ax.set_title(f"{win}: {SPX_NAME}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax = axes[1, col]
    ax.plot(breadth_series(PRED_2008[MAIN_H], MAIN_MODEL).loc[start:b], color="#f4a261", label="Model trained to 2008")
    ax.plot(breadth_series(PRED[MAIN_H], MAIN_MODEL).loc[start:b], color="#1f4e79", label="Model refit every year")
    ax.plot(PRED[MAIN_H]["Base rate"].groupby(level="Date").first().loc[start:b], "k:", label="Base rate")
    if pd.notna(onset):
        ax.axvline(onset, color="#e74c3c", ls="--")
    ax.set_ylabel("Expected bubble share")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
fig.suptitle(f"Signal timing vs drawdown onset ({MAIN_MODEL}, {MAIN_H}M)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

for win, (a, b) in TEST_WINDOWS.items():
    onset = drawdown_onset(a, b)
    if pd.isna(onset):
        continue
    d = PRED[MAIN_H].index.get_level_values("Date")
    snap_date = d[d <= onset - pd.DateOffset(months=6)].max()
    snap = PRED[MAIN_H].xs(snap_date, level="Date").sort_values(MAIN_MODEL, ascending=False).head(5)
    print(f"{win}: top 5 sectors by P(bubble) at {snap_date:%Y-%m} (6 months before the onset)")
    print(snap[[MAIN_MODEL, "RunUp", "PSY_Bubble"]].round(3).to_string())
    print()

## Part 8 — Explainability: what drives P(bubble) in each crisis type?
SHAP ของ Random Forest และ contribution ของ Logit (coef × ค่ามาตรฐาน) สำหรับ sector-months ในปีก่อนวิกฤตแต่ละครั้ง
(SHAP บอกว่า model **ใช้** อะไร ไม่ได้บอกว่าเป็น **สาเหตุ** ของวิกฤต)

In [ ]:
# --- 8.1 SHAP (Random Forest) by crisis type --------------------------------
try:
    import shap
    shap_rows = {}
    for crisis, year in EXPLAIN_YEARS.items():
        if (year, "RandomForest") not in FITTED:
            continue
        model, _ = FITTED[(year, "RandomForest")]
        X = panel.loc[(panel["Year"] == year) & ~panel["Masked"], FEATS_MAIN]
        sv = shap.TreeExplainer(model).shap_values(X)
        sv = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if np.ndim(sv) == 3 else sv)
        imp = np.abs(sv).mean(axis=0)
        shap_rows[crisis] = pd.Series(imp / imp.sum(), index=FEATS_MAIN)
    shap_table = pd.DataFrame(shap_rows)
    ax = shap_table.plot(kind="barh", figsize=(11, 5), color=["#1f4e79", "#e74c3c", "#f4a261", "#2ecc71"])
    ax.set_title("Share of mean |SHAP| by feature — Random Forest, year before each crisis", fontsize=12, fontweight="bold")
    ax.set_xlabel("Share of importance")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("=== SHAP importance share by crisis type ===")
    print(shap_table.round(3).to_string())
except ImportError:
    print("shap not installed -> SHAP skipped (logit contributions below)")

In [ ]:
# --- 8.2 Logit contributions by crisis type (top-10 riskiest sectors) -------
contrib_rows = {}
for crisis, year in EXPLAIN_YEARS.items():
    if (year, "Logit") not in FITTED:
        continue
    model, _ = FITTED[(year, "Logit")]
    scaler, logit = model[0], model[-1]
    X = panel.loc[(panel["Year"] == year) & ~panel["Masked"], FEATS_MAIN]
    p = pd.Series(model.predict_proba(X)[:, 1], index=X.index)
    top = X.loc[p.sort_values(ascending=False).index[:10]]
    z = (top - scaler.mean_) / scaler.scale_
    contrib_rows[crisis] = (z * logit.coef_[0]).mean()
contrib = pd.DataFrame(contrib_rows)
ax = contrib.plot(kind="barh", figsize=(11, 5), color=["#1f4e79", "#e74c3c", "#f4a261", "#2ecc71"])
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Logit contribution to log-odds — 10 riskiest sector-months in the year before each crisis",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Contribution (positive = raises P(bubble))")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(contrib.round(2).to_string())

## Part 9 — Bubble breadth of the S&P 500: "ตอนนี้ตลาดมีส่วนที่เป็น bubble อยู่เท่าไร"
ถ่วงด้วย market cap ของแต่ละ sector:
- **Expected bubble share** = $\sum_i w_i P_i$ (sector ที่ยังไม่ร่วง)
- **High-risk share** = น้ำหนักของ sector ที่ $P \ge$ 35%
- **PSY bubble share** = น้ำหนักของ sector ที่ GSADF flag อยู่ (Layer 1)
- **Already falling** = น้ำหนักของ sector ที่ร่วง ≥20% แล้ว

In [ ]:
# --- 9.1 Breadth series --------------------------------------------------------
B = PRED[MAIN_H].copy()
B["w"] = B["MCap"] / B.groupby(level="Date")["MCap"].transform("sum")
not_falling = ~B["Masked"].astype(bool)
breadth = pd.DataFrame({
    "Expected bubble share": (B["w"] * B[MAIN_MODEL] * not_falling).groupby(level="Date").sum(),
    "High-risk share":       (B["w"] * ((B[MAIN_MODEL] >= HIGH_P) & not_falling)).groupby(level="Date").sum(),
    "PSY bubble share":      (B["w"] * (B["PSY_Bubble"] == 1)).groupby(level="Date").sum(),
    "Already falling":       (B["w"] * B["Masked"].astype(bool)).groupby(level="Date").sum(),
    "Base rate":             B["Base rate"].groupby(level="Date").first()})
spx_b = spx_m.reindex(breadth.index, method="ffill")
crisis_p = regime_m["Crisis"].reindex(breadth.index)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True, gridspec_kw={"height_ratios": [1, 1.4, 0.6]})
ax = axes[0]
ax.plot(spx_b.index, spx_b, color="black", lw=1)
ax.fill_between(spx_b.index, 0, 1, where=(spx_b / spx_b.cummax() - 1) <= -BEAR_DD, color="#e74c3c", alpha=0.15,
                transform=ax.get_xaxis_transform(), label=f">= {BEAR_DD:.0%} below previous peak")
ax.set_yscale("log")
ax.set_ylabel(SPX_NAME)
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
ax = axes[1]
ax.stackplot(breadth.index, breadth["High-risk share"], breadth["Already falling"], colors=["#f4a261", "#bdc3c7"],
             alpha=0.7, labels=[f"High-risk share (P >= {HIGH_P:.0%})", "Already falling"])
ax.plot(breadth.index, breadth["Expected bubble share"], color="#c0392b", lw=1.5, label="Expected bubble share")
ax.plot(breadth.index, breadth["PSY bubble share"], color="#1f4e79", lw=0.8, alpha=0.7, label="PSY bubble share")
ax.plot(breadth.index, breadth["Base rate"], "k:", lw=1, label="Base rate")
ax.set_ylim(0, 1)
ax.set_ylabel("Share of market cap")
ax.legend(loc="upper left", fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
ax = axes[2]
ax.fill_between(crisis_p.index, 0, crisis_p, color=REGIME_COLORS["Crisis"], alpha=0.6)
ax.set_ylabel("HMM P(Crisis)")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
fig.suptitle(f"Bubble breadth of the {SPX_NAME} ({MAIN_MODEL}, {MAIN_H}M, out-of-sample)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- 9.2 Case study: every bear market since 1960 ---------------------------
def bear_markets(px, threshold=BEAR_DD):
    out, peak_v, peak_d, in_bear, trough = [], px.iloc[0], px.index[0], False, None
    for d, v in px.items():
        if not in_bear:
            if v > peak_v:
                peak_v, peak_d = v, d
            elif v / peak_v - 1 <= -threshold:
                in_bear, trough = True, (d, v)
        else:
            if v < trough[1]:
                trough = (d, v)
            if v >= peak_v:
                out.append((peak_d, trough[0], trough[1] / peak_v - 1))
                in_bear, peak_v, peak_d = False, v, d
    if in_bear:
        out.append((peak_d, trough[0], trough[1] / peak_v - 1))
    return out

def value_before(series, date, months):
    s = series.loc[:date - pd.DateOffset(months=months)]
    return s.iloc[-1] if len(s) else np.nan

median_share = breadth["Expected bubble share"].median()
rows = []
for peak, trough, dd in bear_markets(spx_b.dropna()):
    if peak < breadth.index[0] + pd.DateOffset(months=12):
        continue
    e6 = value_before(breadth["Expected bubble share"], peak, 6)
    snap = B.xs(breadth.loc[:peak - pd.DateOffset(months=6)].index[-1], level="Date")
    snap = snap[~snap["Masked"].astype(bool)]
    top = (snap["w"] * snap[MAIN_MODEL]).sort_values(ascending=False).head(3).index
    rows.append({"Peak": f"{peak:%Y-%m}", "Trough": f"{trough:%Y-%m}", "Max DD": f"{dd:.0%}",
                 "Exp. share -12M": value_before(breadth["Expected bubble share"], peak, 12),
                 "Exp. share -6M": e6, "vs median": f"{e6 / median_share:.1f}x",
                 "High-risk -6M": value_before(breadth["High-risk share"], peak, 6),
                 "PSY share -6M": value_before(breadth["PSY bubble share"], peak, 6),
                 "Top sectors -6M": ", ".join(f"{i} ({snap.loc[i, MAIN_MODEL]:.0%})" for i in top)})
case = pd.DataFrame(rows)
print(f"=== {SPX_NAME} bear markets (>= {BEAR_DD:.0%}): breadth BEFORE the peak (median expected share = {median_share:.3f}) ===")
print(case.drop(columns="Top sectors -6M").round(3).to_string(index=False))
print()
print(case[["Peak", "Top sectors -6M"]].to_string(index=False))

In [ ]:
# --- 9.3 Latest reading --------------------------------------------------------
last = breadth.index[-1]
print(f"=== Latest month: {last:%Y-%m} ===")
for k in ["Expected bubble share", "High-risk share", "PSY bubble share", "Already falling", "Base rate"]:
    pct = (breadth[k] <= breadth[k].loc[last]).mean()
    print(f"  {k:22s}: {breadth[k].loc[last]:6.1%}   (historical percentile {pct:.0%})")
print(f"  {'HMM regime':22s}: {regime_m['Regime'].iloc[-1]} (P(Crisis) = {regime_m['Crisis'].iloc[-1]:.2f})")

now = B.xs(last, level="Date").join(panel.xs(last, level="Date")[["R24", "Net24", "Accel"]])
now = now[~now["Masked"].astype(bool)].sort_values(MAIN_MODEL, ascending=False)
now["Sector"] = [SECTOR_OF.get(i, "") for i in now.index]
print(f"\nTop 10 industries by P(bubble) ({MAIN_H}M):")
print(now[["Sector", MAIN_MODEL, "RandomForest", "w", "RunUp", "PSY_Bubble", "R24", "Net24", "Accel"]]
      .rename(columns={"w": "Weight"}).head(10).round(3).to_string())

## Part 10 — Summary

In [ ]:
# --- One-page summary for the report -----------------------------------------
print("=" * 78)
print("  ONE-PAGE SUMMARY — S&P 500 Bubble & Crisis Early-Warning System")
print("=" * 78)
print("LAYER 1 — GSADF")
print(f"  {SPX_NAME}: GSADF {spx_gsadf['GSADF']:.2f} vs CV {spx_gsadf['CV 95%']:.2f}; "
      f"{len(spx_ep)} bubble episodes (>= 3 months)")
print(f"  Sectors rejecting H0: {int(gsadf_table['Reject H0'].sum())}/11; "
      f"led most often: {leaders.index[0] if leaders.max() > 0 else 'n/a'}")
print("LAYER 2 — HMM (3 states)")
print(f"  BIC prefers {sel['BIC'].idxmin()} states; expected duration (days): "
      + ", ".join(f"{r} {d:.0f}" for r, d in duration["Expected days"].items()))
print(f"LAYER 3 — P(bubble), out-of-sample {FIRST_TEST_YEAR}+, early-warning months")
for H in H_LIST:
    r = results[H].loc[MAIN_MODEL]
    print(f"  {H:2d}M {MAIN_MODEL}: AUC {r['AUC']:.3f} [{r['AUC lo']:.3f}, {r['AUC hi']:.3f}] | XS-AUC {r['XS-AUC']:.3f} | "
          f"PR lift {r['PR lift']:.2f} | BSS {r['BSS']:+.3f} [{r['BSS lo']:+.3f}, {r['BSS hi']:+.3f}] | Recall {r['Recall']:.2f}")
e = event_table.loc[(f"{MAIN_H}M", MAIN_MODEL)]
print(f"  Events ({MAIN_H}M): detected {e['Detection rate']:.0%}, median lead {e['Median lead (months)']:.0f} months, "
      f"{e['False alarms / year']:.1f} false alarms / year")
print("MARKET LEVEL")
print(f"  Latest {last:%Y-%m}: expected bubble share {breadth['Expected bubble share'].iloc[-1]:.1%}, "
      f"high-risk share {breadth['High-risk share'].iloc[-1]:.1%}, HMM regime {regime_m['Regime'].iloc[-1]}")
print()
print("Legend:")
print("  AUC     - ranks crash vs no-crash months (0.5 = random)")
print("  XS-AUC  - ranks sectors within the same month (which sector, not when)")
print("  PR lift - precision-recall AUC / base rate (1 = random)")
print("  BSS     - Brier skill vs base rate (> 0 = probability closer to reality)")
print("  Recall  - share of crash months flagged by an alert (P >= 2 x base rate)")

### Limitations
- **Sector ≠ S&P 500**: 48 อุตสาหกรรมของ Ken French ครอบคลุมหุ้นสหรัฐทั้งตลาด (CRSP) ไม่ใช่เฉพาะ S&P 500 และการจัดกลุ่ม 11 sector เป็นการประมาณ GICS
- **จับจังหวะตลาด**: model เก่งเรื่อง "sector ไหนเสี่ยง" (XS-AUC) มากกว่า "เมื่อไร" (pooled AUC) — ตลาดหมีที่ไม่ได้มาจาก sector bubble (เช่น 2008, 2020) จะไม่เห็นสัญญาณ
- **เหตุการณ์ไม่อิสระ**: sector crash พร้อมกันและ label ซ้อนหลายเดือน → ใช้ block bootstrap แล้ว แต่ CI อาจยังแคบเกินจริง
- **HMM** fit ทั้งช่วง (descriptive) แล้วใช้ forward filter; GSADF ใช้ lag 0 และ CV แบบ family-wise 24 เดือน
- **Issuance** ใช้จำนวนบริษัทแทน share issuance (ไม่มี CRSP); drawdown ใช้ราคาสิ้นเดือน
- **Data vintage**: Ken French / FRED / Shiller เป็นข้อมูลฉบับปรับปรุงล่าสุด ไม่ใช่ข้อมูล real-time

**References**
- Greenwood, R., Shleifer, A., & You, Y. (2019). Bubbles for Fama. *Journal of Financial Economics*, 131(1), 20–43.
- Phillips, P.C.B., Shi, S., & Yu, J. (2015). Testing for multiple bubbles. *International Economic Review*, 56(4), 1043–1078.
- Phillips, P.C.B., & Shi, S. (2020). Real time monitoring of asset markets: Bubbles and crises. *Handbook of Statistics*, 42, 61–80.
- Gu, X. (2022). *Early-Warning Alert Systems for Financial-Instability Detection: An HMM-Driven Approach*. PhD thesis, Western University.
- Martin, I., & Shi, R. (2025). Forecasting crashes with a smile. LSE FMG Discussion Paper 936.